# Lesson 49 · The DCT-II — twisting 40 correlated numbers into 13 independent coefficients: a pure-NumPy discrete cosine transform by hand

**Goal**: first understand why MFCCs need one more step, a DCT-II, after the log-mel, then implement it from scratch and "twist" the correlated mel dimensions into more independent coefficients.

> **Motivation first, formula second**: first see why it decorrelates and compresses, then write a pure-NumPy DCT-II.

🔗 Aurora connection: `aurora.audio.mfcc.dct_ii()` is the answer key for this lesson, and the final MFCC pipeline calls it directly.

← **Previous lesson** · [L48 · Time–frequency plots compared](L48_visual_stft.ipynb)

> Last lesson covered **time–frequency plots compared**: a side-by-side visual comparison of the linear, mel and log-mel spectrograms.
> This lesson explores **the DCT-II, the discrete cosine transform**.

## This lesson's story: why do MFCCs need one more transform after the log-mel?

Imagine you've recorded a conversation and computed 80-dimensional log-mel features. Those 80 numbers sound complete — but there's a catch: neighboring mel channels overlap by about 50%. That means number 10 and number 11 are saying almost the same thing (correlation can reach 0.9+).

Feeding highly correlated features into a neural network is like telling it "these two things matter" — when they're really one thing. It wastes capacity and makes optimization harder.

**The DCT-II (discrete cosine transform) solution**: "rotate" the N correlated numbers into another coordinate system so that most of the information piles up in the first few dimensions and the rest are close to zero. It's like PCA — except the DCT's basis is a fixed set of cosines that doesn't have to be learned from data.

**The orthonormal DCT-II formula**:

$$X[k] = w(k)\sum_{n=0}^{N-1} x[n]\cos\!\left(\frac{\pi k(2n+1)}{2N}\right)$$

where $w(0)=\frac{1}{\sqrt{N}}$ and $w(k>0)=\sqrt{\frac{2}{N}}$.

**Why keep the first 13 dimensions (MFCC-13)?**
The DCT concentrates energy in the low-order coefficients — the first 13 represent the slowly varying shape of the vocal tract (the phoneme), while higher coefficients correspond to fast ripples (unstable noise). Truncating = reducing dimensions + removing noise.

Your task in this lesson: implement `dct_ii(x)` — pure NumPy, with an error < 1e-10 against `scipy.fft.dct(norm='ortho')`.

## Think first: why not just feed the 40 mel dimensions to the model?

Before reading on to the DCT formula, think on your own for 30 seconds:

1. Neighboring mel filters overlap by about 50%. Are the values in dimensions 10 and 11 "two independent things," or "one thing said twice"?
2. If these 40 numbers are packed with repeated information and fed straight to the model, the model has to spend effort discovering "these two are really the same" — is that helping or getting in the way?

Think those two points through and you've grasped the motivation for this lesson: **the DCT isn't mathematical showing off — it decorrelates 40 highly correlated numbers and compresses them into a dozen or so nearly independent coefficients**.

## 🤔 Why did engineers invent this?
- **What happens without it?** Use the 40-dimensional log-mel directly and neighboring dimensions have correlations as high as 0.9+, which amounts to telling the model the same thing over and over — wasting capacity and slowing optimization. Models that "assume independent features," such as GMMs or Euclidean distance, may even break down.
- **What real problem does it solve?** With a **fixed** set of cosine bases it rotates the correlated information and concentrates it in the first few dimensions (like a PCA that needs no training); the later dimensions are close to 0 and can be safely truncated → reducing dimensions and removing noise at the same time.
- **Where will it show up again?** It's the last step of the L50 MFCC pipeline; keeping the first 13 dimensions gives the classic MFCC-13.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

## 0. First, FFT vs DCT-II: they're not the same thing

| | FFT / DFT | DCT-II |
|---|---|---|
| Basis functions | complex exponentials | cosines |
| Boundary assumption | circular / periodic extension | even symmetric extension |
| Output | complex spectrum | real cepstral coefficients |
| Typical use | spectrum analysis | decorrelation / compression |
| Aurora | `aurora.audio.transforms.fft` | `aurora.audio.mfcc.dct_ii` |

The DCT-II comes after the log-mel because neighboring mel channels are highly correlated;
the DCT rotates the repeated information into fewer coordinates.
If you've forgotten why we do this step, look back at the motivation appendix at the end of L48.

## ⚠️ Common misconceptions
> Don't think of the DCT as "another kind of FFT." It's really a **decorrelating rotation with a fixed basis**: the FFT uses a complex-exponential basis for spectrum analysis, outputs complex numbers and assumes a periodic extension of the signal; the DCT-II uses a pure cosine basis, outputs real numbers and assumes an even symmetric extension, and its aim is to push correlated energy into the low-order coefficients. The table above compares the two row by row — remember: "both orthogonal transforms, completely different jobs."

## 1. The DCT-II formula (orthonormal form)

aurora uses the same **orthonormal** convention as `scipy.fft.dct(..., norm='ortho')`:

```
X[k] = s_k * sum_{n=0}^{N-1} x[n] * cos(pi * (2n+1) * k / (2N))

s_0 = sqrt(1/N)      <- special scaling for the k=0 DC coefficient
s_k = sqrt(2/N)      <- every coefficient with k > 0
```

Compared with the "multiply by 2, no normalization" version, the orthonormal version has three advantages:

- **The matrix is orthogonal**: `M_ortho @ M_ortho.T = I`, so the inverse transform is just the transpose (no dividing by 2N)
- **Energy is preserved** (Parseval's theorem): `||X||^2 = ||x||^2`, which keeps the numbers more stable during compression/reconstruction
- **Compatible with the scipy / numpy ecosystem**: aurora's output can be compared directly against librosa's reference values

**Why treat k=0 separately?**

The k=0 basis vector is the constant 1, whose L2 norm is `sqrt(N)` (while the basis vectors for k>0 have norm `sqrt(N/2)`).
To make all basis vectors orthonormal on the same scale, k=0 needs an extra division by `sqrt(2)`, i.e. `s_0 = sqrt(1/N)`.

## 1.5 Deep dive: the L2 norms of the basis vectors: why √N and √(N/2)?

This is where students get stuck most often. Let's work it out step by step with **a concrete example, N=4**.

### k=0 (the DC component; the basis vector is the constant 1)

$$\text{basis vector} = \begin{pmatrix} 1 \\ 1 \\ 1 \\ 1 \end{pmatrix}, \quad L2 \text{ norm} = \sqrt{1^2 + 1^2 + 1^2 + 1^2} = \sqrt{4} = 2 = \sqrt{N}$$

✓ For N=4, the k=0 basis vector's norm is indeed √4 = 2.

### k=1 (the lowest-frequency cosine; the basis vector is made of cos values)

Element n of the DCT-II basis vector is:
$$\cos\left(\frac{\pi \cdot 1 \cdot (2n+1)}{2 \cdot 4}\right) = \cos\left(\frac{\pi(2n+1)}{8}\right)$$

For n=0,1,2,3:

$$\begin{align}
n=0: &\quad \cos\left(\frac{\pi}{8}\right) \approx 0.9239 \\
n=1: &\quad \cos\left(\frac{3\pi}{8}\right) \approx 0.3827 \\
n=2: &\quad \cos\left(\frac{5\pi}{8}\right) \approx -0.3827 \\
n=3: &\quad \cos\left(\frac{7\pi}{8}\right) \approx -0.9239
\end{align}$$

basis vector = $\begin{pmatrix} 0.9239 \\ 0.3827 \\ -0.3827 \\ -0.9239 \end{pmatrix}$

L2 norm:
$$\sqrt{0.9239^2 + 0.3827^2 + 0.3827^2 + 0.9239^2} = \sqrt{2 \cdot (0.9239^2 + 0.3827^2)}$$

Numerically this does equal √2 = √(N/2). But "the numbers happen to match" and "it's proven for every N and every k" are two different things — below we **actually derive** it, instead of stopping at "it can be shown."

### Full derivation: $\sum_{n=0}^{N-1}\cos^2\left(\frac{\pi k(2n+1)}{2N}\right) = \frac{N}{2}$ (for k > 0)

**Analogy first**: imagine N points scattered evenly around a clock face, each one a direction (an angle). If these directions are spread evenly all the way around, then adding them up as arrows placed tip to tail brings you exactly back to where you started — the total is zero. This "go once around and everything cancels" intuition is the heart of the derivation below.

**Step one: split cos² with the power-reduction identity**

The power-reduction identity from school:
$$\cos^2\theta = \frac{1+\cos(2\theta)}{2}$$

Write $\theta_n = \dfrac{\pi k(2n+1)}{2N}$ and substitute into the sum we want:

$$\sum_{n=0}^{N-1}\cos^2(\theta_n) = \sum_{n=0}^{N-1}\frac{1+\cos(2\theta_n)}{2} = \underbrace{\frac{N}{2}}_{N\text{ copies of }\frac12} + \frac{1}{2}\sum_{n=0}^{N-1}\cos(2\theta_n)$$

All we need to show is that the second part, $\sum\cos(2\theta_n)=0$, and the answer is exactly N/2.

**Step two: use Euler's formula (recall L06) to turn the cosine sum into a geometric series**

$2\theta_n = \dfrac{\pi k(2n+1)}{N} = \dfrac{\pi k}{N} + n\cdot\dfrac{2\pi k}{N}$, an arithmetic sequence (common difference $d=\frac{2\pi k}{N}$).

From L06's Euler formula, $\cos\phi=\text{Re}(e^{i\phi})$, so:

$$\sum_{n=0}^{N-1}\cos(2\theta_n) = \text{Re}\left[e^{i\pi k/N}\sum_{n=0}^{N-1}\left(e^{i2\pi k/N}\right)^n\right]$$

Inside the brackets is a geometric series with ratio $r=e^{i2\pi k/N}$:

$$\sum_{n=0}^{N-1} r^n = \frac{r^N-1}{r-1}\quad(r\neq 1)$$

Substitute $r^N = \left(e^{i2\pi k/N}\right)^N = e^{i2\pi k} = 1$ (k is an integer, so that's a whole number of turns — back to the start!).

The numerator $r^N-1=0$, so as long as $r\neq 1$ (that is, k isn't a multiple of N, which always holds for $k=1,\dots,N-1$), the whole sum is **0** — exactly the maths behind the clock-face analogy: N arrows spread evenly all the way around cancel tip to tail.

**Step three: put it together**

$$\sum_{n=0}^{N-1}\cos^2(\theta_n) = \frac{N}{2} + \frac{1}{2}\cdot 0 = \frac{N}{2}$$

So the L2 norm $=\sqrt{N/2}$. That's the full derivation — no longer "it can be shown," but worked out step by step from the power-reduction identity and the geometric-series formula. The code below checks steps one, two and three:

### The general case (checked by the code below)

In [ ]:
# Check each step of the derivation: the cos² split + the geometric series summing to 0
N, k = 8, 1
n = np.arange(N)
theta = np.pi * k * (2 * n + 1) / (2 * N)

lhs = np.sum(np.cos(theta) ** 2)                        # compute ∑cos²(θ) directly
rhs_step1 = N / 2 + 0.5 * np.sum(np.cos(2 * theta))     # the two terms after the split
geo_sum = np.sum(np.exp(1j * 2 * np.pi * k * n / N))     # geometric series ∑ r^n

print(f"direct ∑cos²(θ)            = {lhs:.6f}")
print(f"split N/2 + (1/2)∑cos(2θ)  = {rhs_step1:.6f}  (should equal the line above)")
print(f"geometric series ∑r^n (r=e^(i2πk/N)) = {geo_sum:.2e}  (should be ≈0: a whole number of turns)")
print(f"so ∑cos²(θ) = N/2 = {N/2}")

In [ ]:
# Check: the L2 norms of the basis vectors for N=4
N = 4
k_vals = np.arange(N)
n_vals = np.arange(N)

# Build the N×N cosine matrix
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))

print("=" * 60)
print(f"Basis vectors and norms for N = {N}:")
print("=" * 60)

for k in range(N):
    basis_vec = M[k]
    norm = np.linalg.norm(basis_vec)

    if k == 0:
        expected = np.sqrt(N)
        print(f"\nk={k} (constant basis, DC component)")
        print(f"  vector: {basis_vec}")
        print(f"  L2 norm: {norm:.6f}")
        print(f"  expected √N = √{N} = {expected:.6f} ✓")
    else:
        expected = np.sqrt(N / 2)
        print(f"\nk={k} (cosine basis)")
        print(f"  vector: {np.round(basis_vec, 4)}")
        print(f"  L2 norm: {norm:.6f}")
        print(f"  expected √(N/2) = √{N/2} = {expected:.6f} ✓")

print("\n" + "=" * 60)
print("Summary:")
print(f"  • norm of the k=0 basis vector = √N = √{N} = {np.sqrt(N):.6f}")
print(f"  • norm of the k>0 basis vectors = √(N/2) = √{N/2} = {np.sqrt(N/2):.6f}")
print("  • the two values differ, so the normalization weights must differ too!")
print("=" * 60)

In [ ]:
# Demo: look at the DCT-II cosine basis vectors (N=8)
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
# M[k, n] = cos(pi*(2n+1)*k / (2N))
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
print("cosine matrix M, shape:", M.shape)
print("row 0 (k=0, constant basis):", np.round(M[0], 4))
print("row 1 (k=1, lowest-frequency cosine):", np.round(M[1], 4))
print("row 4 (k=4, higher-frequency cosine):", np.round(M[4], 4))

## 2. DCT vs DFT: a more efficient representation for real-valued signals

The DFT assumes the signal repeats with period N; when the two ends of the signal don't join up, you get a "Gibbs phenomenon" (spectral leakage).
The DCT-II is equivalent to taking the real part of the DFT of an even symmetric extension of the signal, so there's no jump at the boundary and far less leakage, and the output is purely real.

Orthogonality check: the diagonal of `M @ M.T` is N/2 (k>0) or N (k=0) and the off-diagonal entries are zero,
which shows the cosine basis vectors are pairwise orthogonal (this is where the comment "should be N/2=4 or N=8" in the orthogonality-check code below comes from).
Equivalently, the DCT matrix D with its normalization factors (scale[0]=√(1/N), scale[k>0]=√(2/N)) satisfies D @ D.T = I,
and that's exactly why the transform can be inverted exactly (the inverse is the transpose).

## 2.1 Extra: what is an "even symmetric extension"? Why is the DCT = the real part of an even-symmetric DFT?

An **even symmetric extension** "mirror-flips" a finite-length signal so that it looks **symmetric about its midpoint**.

### Example: the even symmetric extension of [1, 2, 3, 4]

Original signal: `x = [1, 2, 3, 4]`

**The first kind of extension** (mirrored about the last sample itself):
```
original: [1, 2, 3, 4]
extended: [1, 2, 3, 4, 3, 2]  (length = 2N-2 = 6 — symmetric about the last sample; this is not the DCT-II one)
```

**The extension the DCT-II uses** (mirrored between the last sample and the next, virtual sample):
```
original:     [1,    2,    3,    4]
mapped angles: π/8   3π/8  5π/8  7π/8
extended:     [1,    2,    3,    4,    4,    3,    2,    1] (length = 2N = 8)
```

Key point: the extension has **2N elements**, forming a "mirror-symmetric" sequence.

### Why is the DCT-II ≈ the real part of an even-symmetric DFT?

The DFT of an even symmetric sequence has a special property:
- because of the symmetry, the imaginary part is naturally zero
- the real part is exactly a cosine transform

That matches the fact that the DCT-II's basis functions are **cos** (not complex exponentials $e^{i\omega}$).

The code below demonstrates the relationship with a small N=4 example:

In [ ]:
# Demo: the idea that the DCT-II matches the real part of an even-symmetric DFT
N = 4
x = np.array([1.0, 2.0, 3.0, 4.0])
print("original signal (length N=4):", x)

# Method 1: compute the DCT-II directly with the cosine matrix (no weights, just the basic form)
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
dct_direct = M @ x
print("\nMethod 1 - direct DCT-II result:", np.round(dct_direct, 4))

# Method 2: even symmetric extension → real part of the DFT
# Extension: [1, 2, 3, 4] → [1, 2, 3, 4, 3, 2] (length 2N-2) — symmetric about the last sample
# The exact DCT-II extension samples on a (2N) half-sample grid, which amounts to
# the "periodic extension" [1, 2, 3, 4, 4, 3, 2, 1]
x_extended = np.concatenate([x, x[-2:0:-1]])  # [1,2,3,4,3,2]
print("\n[Note] the standard even symmetric extension:", x_extended)

# The actual DCT-II uses a finer sampling, but this approximation already shows the DFT property
# Take the FFT of an even symmetric signal and the imaginary part comes out tiny
from scipy.fft import fft
X_fft = fft(x_extended)
print("FFT result (real part):", np.round(X_fft.real, 4))
print("FFT result (imag part):", np.round(X_fft.imag, 4), "← should be close to 0 (even symmetry)")

print("\n[Core idea]")
print("  At heart, the DCT-II gives the signal an even symmetric extension and takes the real part of the relevant DFT frequencies.")
print("  The payoff: it avoids the FFT's 'circular convolution' boundary problem, and the output is purely real.")

In [ ]:
# Demo: check the orthogonality of the cosine matrix
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
G = M @ M.T  # Gram matrix; in theory the diagonal is N/2 (k>0) or N (k=0)
print("diagonal of M @ M.T (should be N/2=4 or N=8):")
print(np.round(np.diag(G), 6))
print("largest off-diagonal absolute value (should be ≈0):", np.max(np.abs(G - np.diag(np.diag(G)))))

## 2.2 Deep dive: Parseval's theorem: why does an orthogonal transform preserve energy?

**Parseval's theorem** (for orthogonal transforms):
$$\|X\|^2 = \|x\|^2 \quad \text{i.e.} \quad \sum_{k=0}^{N-1} X[k]^2 = \sum_{n=0}^{N-1} x[n]^2$$

**Why does it hold?** If $D$ is an orthogonal matrix ($D D^T = I$), then:

$$\|X\|^2 = \|D \cdot x\|^2 = (Dx)^T (Dx) = x^T D^T D x = x^T I x = \|x\|^2$$

**Physical meaning**:
- the "total energy" of the signal before the transform = the "total energy" of the coefficients after it
- no energy is lost, and none appears out of nowhere
- this matters for compression: when you truncate the high-order coefficients, the loss of total energy isn't exaggerated

The code below checks it:

In [ ]:
# Check Parseval's theorem: ||X||^2 = ||x||^2
N = 16
x = np.random.default_rng(50).standard_normal(N)

# Build the orthogonal DCT matrix
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
scale = np.full(N, np.sqrt(2.0 / N))
scale[0] = np.sqrt(1.0 / N)
D = scale[:, None] * M

# Apply the transform
X = D @ x

# Energy (squared L2 norm)
energy_before = np.sum(x**2)
energy_after = np.sum(X**2)

print("=" * 60)
print("Parseval's theorem check: an orthogonal transform preserves energy")
print("=" * 60)
print(f"signal energy ||x||²       = {energy_before:.8f}")
print(f"transformed energy ||X||²  = {energy_after:.8f}")
print(f"relative error             = {abs(energy_before - energy_after) / energy_before:.2e}")
print("\n✓ Energy preserved! That's why the DCT matrix has to be orthogonal.")

## 2.3 Deep dive: from "rows with different norms" to "why we need different weights"

From the code above you can see that the k=0 row vector has norm √N, while every k>0 row vector has norm √(N/2).

What happens if we **transform with M directly (no weights)**?

$$G = M \cdot M^T$$

On the diagonal:
- row 0: $M[0] \cdot M[0] = (\sqrt{N})^2 = N$
- row k (k>0): $M[k] \cdot M[k] = (\sqrt{N/2})^2 = N/2$

Look, the diagonal isn't uniform! Row 0 isn't on the same "scale" as the other rows.

**To make all basis vectors orthonormal on the same scale**, we need to "rescale" each row:

$$D[k, :] = \frac{M[k, :]}{\text{norm of } M[k, :]} \quad \text{(normalization)}$$

That way every row has norm 1. And the normalization factors are:

$$w(k) = \begin{cases}
\frac{1}{\sqrt{N}} & k=0 \quad \text{(because the original norm is } \sqrt{N} \text{)} \\
\frac{1}{\sqrt{N/2}} = \sqrt{\frac{2}{N}} & k>0 \quad \text{(because the original norm is } \sqrt{N/2} \text{)}
\end{cases}$$

**But won't the weights mess up the "off-diagonal entries" too?** This is where many students get stuck — so far we've only handled the "diagonal" (each row's inner product with itself). What about the inner product of "row 0" and "row 1" (an off-diagonal entry)? Could multiplying by the weights turn it from 0 into something nonzero?

The answer: **no**. The reason is simple — **if a sum is 0 to begin with, multiplying every term by the same nonzero constant still gives 0** (0 times anything is still 0). As a formula:

$$D_{0,n}\cdot D_{1,n} = \big(w(0)\,M_{0,n}\big)\big(w(1)\,M_{1,n}\big) = w(0)\,w(1)\cdot M_{0,n}M_{1,n}$$

Summing over n:

$$\sum_{n=0}^{N-1} D_{0,n}D_{1,n} = w(0)\,w(1)\sum_{n=0}^{N-1} M_{0,n}M_{1,n} = w(0)\,w(1)\cdot 0 = 0$$

Here $\sum_n M_{0,n}M_{1,n}=0$ is a fact that already holds **before the weights are applied** (the orthogonality check in section 2 above verified it: the off-diagonal of the unweighted $M@M.T$ is already 0 — that's the orthogonality of the cosine basis itself, nothing to do with the weights). The weights $w(0)w(1)$ are just a constant factor pulled out front, and they can't turn 0 into something nonzero.

So: **the weights change only the diagonal (rescaling N or N/2 to a uniform 1) and don't touch the off-diagonal at all (0 before, still 0 after)**. That's why "applying the weights" makes the matrix fully orthogonal (not only is the diagonal 1, the off-diagonal stays 0).

The code below first works out the inner product of row k=0 and row k=1 by hand (before and after weighting), then shows the whole matrix:

In [ ]:
# Hand check: does the inner product of row k=0 and row k=1 stay 0 before and after weighting?
N = 8
n = np.arange(N)
M0 = np.cos(np.pi * 0 * (2 * n + 1) / (2 * N))   # row k=0 (all ones)
M1 = np.cos(np.pi * 1 * (2 * n + 1) / (2 * N))   # row k=1

inner_unweighted = np.sum(M0 * M1)
print(f"before weighting: M_0 · M_1 = {inner_unweighted:.2e}  (should be ≈0: the cosine basis is itself orthogonal)")

w0 = np.sqrt(1.0 / N)
w1 = np.sqrt(2.0 / N)
D0 = w0 * M0
D1 = w1 * M1
inner_weighted = np.sum(D0 * D1)
print(f"after weighting: D_0 · D_1 = w0*w1*(M_0·M_1) = {w0*w1:.4f} * ({inner_unweighted:.2e}) = {inner_weighted:.2e}  (still ≈0)")

In [ ]:
# Demo: use weights to bring rows of different norms to norm 1
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))

# Original Gram matrix (no weights)
G_unscaled = M @ M.T
print("[No weights] diagonal of M @ M.T:")
print(np.round(np.diag(G_unscaled), 2))
print("  → k=0 is", np.round(np.diag(G_unscaled)[0], 2), ", k>0 is", np.round(np.diag(G_unscaled)[1], 2))
print()

# Apply the weights: scale[0]=sqrt(1/N), scale[k>0]=sqrt(2/N)
scale = np.full(N, np.sqrt(2.0 / N))
scale[0] = np.sqrt(1.0 / N)
D = scale[:, None] * M  # multiply each row by its weight

# Gram matrix after weighting
G_scaled = D @ D.T
print("[After weighting] diagonal of D @ D.T:")
print(np.round(np.diag(G_scaled), 6))
print("  → every diagonal entry is now 1!")
print()

print("[Key finding] off-diagonal entries (inner products of two rows):")
print("largest absolute value =", np.max(np.abs(G_scaled - np.eye(N))))
print("  → should be ≈0 (the condition for an orthogonal matrix)")

## 2.4 Counterexample: what happens if k=0 doesn't get its special weight?

By now you might be thinking: since building an orthogonal matrix is this much trouble (k=0 needs separate handling), is it worth it? If we cut corners and use the same weight $\sqrt{2/N}$ for every k (k=0 included), what goes wrong?

**An analogy**: picture a ruler where most of the marks are 1 cm apart, but one gap was misprinted as 1.4 cm. Measure things with it and most of the time you're fine, but whenever you cross that gap the reading is systematically off. The k=0 weight of the DCT is that "misprinted mark" — without special handling, the "energy" you measure is systematically inflated.

Concretely, leaving out the special k=0 weight breaks three things:
1. **The matrix is no longer orthogonal** ($D D^T \neq I$: the k=0 row's inner product with itself becomes 2 instead of 1)
2. **The inverse transform can no longer be the transpose** — you'd have to compute a true matrix inverse $D^{-1}$, so instead of just a matrix–vector product you first pay the larger cost of inverting
3. **Numerical accuracy drops** — if you keep cutting corners and use the transpose as the inverse anyway, you introduce a fixed, systematic error (not random noise, but a predictable bias)

The code below puts the "correct version" (k=0 uses $\sqrt{1/N}$) and the "lazy version" (k=0 also uses $\sqrt{2/N}$) side by side:

In [ ]:
# Counterexample: what happens without the special k=0 weight?
N = 16
n = np.arange(N)
k = np.arange(N)
M = np.cos(np.pi * np.outer(k, 2 * n + 1) / (2 * N))

# Correct version: k=0 uses sqrt(1/N)
scale_correct = np.full(N, np.sqrt(2.0 / N))
scale_correct[0] = np.sqrt(1.0 / N)
D_correct = scale_correct[:, None] * M

# Lazy version: every k uses sqrt(2/N) (k=0 included)
scale_wrong = np.full(N, np.sqrt(2.0 / N))
D_wrong = scale_wrong[:, None] * M

print("=" * 70)
print("Thing one: is the matrix still orthogonal?")
print("=" * 70)
G_correct = D_correct @ D_correct.T
G_wrong = D_wrong @ D_wrong.T
print(f"correct version, diagonal of D @ D.T (should be all 1): {np.round(np.diag(G_correct), 4)}")
print(f"lazy version, diagonal of D @ D.T: {np.round(np.diag(G_wrong), 4)}")
print(f"  → the k=0 row's inner product with itself became {G_wrong[0, 0]:.2f}, not 1! The matrix is no longer orthogonal.")

print()
print("=" * 70)
print("Things two and three: the cost and accuracy of the inverse")
print("=" * 70)
rng = np.random.default_rng(3)
x = rng.standard_normal(N)

X_wrong = D_wrong @ x  # forward transform with the lazy matrix

# If we lazily keep using "the transpose as the inverse" (pretending the matrix is still orthogonal)
x_rec_by_transpose = D_wrong.T @ X_wrong
err_transpose = np.max(np.abs(x_rec_by_transpose - x))

# Do it properly: compute the true inverse matrix
D_wrong_inv = np.linalg.inv(D_wrong)
x_rec_by_inverse = D_wrong_inv @ X_wrong
err_inverse = np.max(np.abs(x_rec_by_inverse - x))

print(f"error of the lazy transpose reconstruction: {err_transpose:.4f}  ← clearly not 0! A systematic bias (the k=0 component is scaled twice)")
print(f"error of the proper inverse reconstruction: {err_inverse:.2e}  ← correct, but you paid the extra cost of inverting")
print()

X_correct = D_correct @ x
x_rec_correct = D_correct.T @ X_correct
err_correct = np.max(np.abs(x_rec_correct - x))
print(f"compare: the correct version (an orthogonal matrix) reconstructs with the transpose directly, error = {err_correct:.2e}  ← free and more accurate!")

## 3. Decorrelation: from mel energies to cepstral coefficients

Neighboring channels of a mel filterbank overlap by about 50%, so the Pearson correlation between neighboring energy values can exceed 0.9.
The DCT-II projects these correlated features onto an orthogonal cosine basis: the low-order coefficients (k=0..12) capture the overall spectral shape of the speaker/phoneme,
while the high-order coefficients (k>=13) capture detail and are barely correlated — which makes classifiers built around Euclidean distance (GMMs, DTW, neural networks) more efficient.
Usually only the first 13 coefficients are kept: MFCC-13.

**Feeling the effect of "decorrelation" in numbers** (with real speech frames as the reference):

| Feature pair | Before the DCT (mel energy) | After the DCT (MFCC) |
|:---|:---:|:---:|
| neighboring coefficients r(0,1) | ≈ 0.91 | ≈ 0.02 |
| coefficients 2 apart r(0,2) | ≈ 0.77 | ≈ 0.01 |
| coefficients 5 apart r(0,5) | ≈ 0.43 | < 0.01 |

In mel space, neighboring bins are highly correlated because the filters overlap; after the DCT the coefficients are approximately orthogonal and the covariance matrix is close to diagonal.
That means models that assume independent features, such as GMMs and Euclidean-distance classifiers, can work effectively on MFCCs directly, without extra whitening.

**A quick aside: what is a "covariance matrix"?** Say you've collected M frames of speech, each an N-dimensional mel-energy vector, and stacked them into an $M\times N$ matrix $X$ (one frame per row). First subtract each column's own mean ("centering"); the covariance matrix is then defined as:

$$\text{Cov} = \frac{1}{M}X^\top X \quad (N\times N \text{ matrix})$$

Its **diagonal** holds each dimension's own variance (how "spread out" it is), and its **off-diagonal** entry $(i,j)$ is the covariance of dimensions $i$ and $j$ — the bigger it is, the more those two dimensions "rise and fall together" (are correlated). The large off-diagonal entries in the covariance matrix of mel energies are exactly the mathematical way of saying "neighboring mel bins are highly correlated."

**Why can an orthogonal transform diagonalize it?** Let $Y = XD^\top$ be the features after the DCT ($D$ is our orthogonal DCT matrix). Then:

$$\text{Cov}_Y = \frac{1}{M}Y^\top Y = \frac{1}{M}(XD^\top)^\top(XD^\top) = \frac{1}{M}D X^\top X D^\top = D\,\text{Cov}_X\,D^\top$$

This is exactly the "similarity transform by an orthogonal matrix" from linear algebra. If $D$ happened to be the eigenvector matrix of $\text{Cov}_X$ (i.e. the PCA solution), this expression would diagonalize it exactly. The DCT's cosine basis isn't a set of eigenvectors learned from data, but for signals where "neighbors are highly correlated, like a smooth AR(1) process" (which is exactly how mel energies are structured), the DCT basis is a very good approximation to the PCA eigenvectors — which is why, instead of redoing PCA for every dataset, a fixed DCT basis gets close to diagonalizing the covariance too.

The demo below uses an AR(1) process to simulate correlated mel energies and shows how the correlation structure changes before and after the DCT:

## 3.1 Extra: seeing is believing: why does "50% filter overlap" lead to "0.9+ output correlation"?

We said earlier that neighboring channels of a mel filterbank overlap by about 50% — a rather abstract statement. Let's draw it now, and then use a toy signal to check the cause and effect "overlap → correlation" for ourselves.

**An analogy**: picture two people in the same room, each recording with their own microphone, but with half of the microphones' pickup ranges overlapping. Whenever anyone in the room speaks, both microphones will very likely record much the same change in volume — because they're "listening" to a large shared part of the space. That's the intuition for "overlap" causing "correlation": both filters' outputs respond to the energy in the same stretch of frequencies, so when that energy changes, both outputs change together.

The code below draws 4 neighboring triangular mel filters (each neighboring pair overlapping by 50%), then passes a toy spectrum with slight random jitter (simulating the natural fluctuation of a real speech spectral envelope) through them, and looks at the correlation between the outputs:

In [ ]:
import matplotlib.pyplot as plt

# Build 4 neighboring triangular filters, with neighbors overlapping by 50%
freqs = np.linspace(0, 4000, 400)  # frequency axis (Hz)


def triangular_filter(freqs, left, center, right):
    resp = np.zeros_like(freqs)
    rising = (freqs >= left) & (freqs <= center)
    falling = (freqs > center) & (freqs <= right)
    resp[rising] = (freqs[rising] - left) / (center - left)
    resp[falling] = (right - freqs[falling]) / (right - center)
    return resp


centers = [1000, 1500, 2000, 2500]  # filter center frequencies
half_width = 500                     # half-width = center spacing, guaranteeing 50% overlap between neighbors
filters = [triangular_filter(freqs, c - half_width, c, c + half_width) for c in centers]

plt.figure(figsize=(8, 3))
for i, f in enumerate(filters):
    plt.plot(freqs, f, label=f"Filter {i} (center {centers[i]} Hz)")
plt.title("Neighboring mel filters overlap by 50%")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Response weight")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Toy signal: a spectral envelope that drifts slowly over "time" (simulating the natural wobble in position/level of real speech formants)
rng = np.random.default_rng(0)
outputs = []
for _trial in range(400):
    peak_center = 1700 + rng.normal(0, 150)   # the formant position drifts slightly
    amplitude = 5 + rng.normal(0, 0.8)         # the overall loudness fluctuates slightly
    spectrum = amplitude * np.exp(-0.5 * ((freqs - peak_center) / 1400) ** 2)
    spectrum += rng.normal(0, 0.02, size=freqs.shape)  # a little background noise
    filt_energy = [np.sum(spectrum * f) for f in filters]  # each filter's output = a weighted sum
    outputs.append(filt_energy)
outputs = np.array(outputs)  # (400, 4)

print("Correlation matrix of the 4 filter outputs:")
print(np.round(np.corrcoef(outputs.T), 3))
print("\n[Reading it] filters 0 and 1 have centers just 500 Hz apart and share half a band ([1000,1500] Hz),")
print("so whenever the envelope drifts they almost always rise and fall together → correlation ≈ 0.97 (very high).")
print("Filters 0 and 3 have centers 1500 Hz apart and don't overlap at all, and their correlation is clearly lower —")
print("more overlap → more shared frequency energy → more correlated outputs: that's the cause and effect behind '50% overlap → high correlation'.")

## 3.2 Deep dive: computing the correlation coefficients: before vs after the DCT

You may wonder where a number in the table like r(0,1)≈0.91 comes from. Let's work it out with the Pearson correlation formula:

$$r(X, Y) = \frac{\text{cov}(X, Y)}{\sigma_X \sigma_Y} = \frac{\sum_i (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum_i (X_i - \bar{X})^2} \sqrt{\sum_i (Y_i - \bar{Y})^2}}$$

The code below generates simulated mel energies (with an AR(1) process, Φ=0.9, standing for high correlation between neighbors), then:
1. computes the correlation coefficients **before the DCT**
2. applies the DCT
3. computes the correlation coefficients **after the DCT**
4. shows how the covariance matrix changes (the "diagonalizing" effect):

In [ ]:
# Demo: how the correlations change before and after the DCT
from scipy.stats import pearsonr

# Generate M correlated mel feature vectors (each of length N)
rng = np.random.default_rng(123)
N = 26   # number of mel bins
M = 1000  # collect 1000 frames of speech to compute the correlations

mel_features = np.zeros((M, N))  # (M, N): M frames, N dimensions each

# Simulate mel energies with an AR(1) process: neighboring bins correlated as high as 0.9
for frame_idx in range(M):
    mel_features[frame_idx, 0] = rng.standard_normal()
    for bin_idx in range(1, N):
        mel_features[frame_idx, bin_idx] = (
            0.9 * mel_features[frame_idx, bin_idx - 1] +
            np.sqrt(1 - 0.9**2) * rng.standard_normal()  # stationary AR(1): every bin has variance 1
        )

# Compute the Pearson correlations (before the DCT)
print("=" * 70)
print("[Before the DCT] correlations of the mel energies:")
print("=" * 70)
r_before = np.corrcoef(mel_features.T)  # (N, N) correlation matrix
print(f"r(0, 1) = {r_before[0, 1]:.4f}  (neighboring bins, should be about 0.9)")
print(f"r(0, 2) = {r_before[0, 2]:.4f}  (2 apart, should be about 0.9²≈0.8)")
print(f"r(0, 5) = {r_before[0, 5]:.4f}  (5 apart, should be about 0.9⁵≈0.6)")
print("\n[Structure of the covariance matrix]")
print("  large off-diagonal entries → features highly correlated → the model wastes capacity")

# Apply the DCT
from aurora.audio.mfcc import dct_ii as aurora_dct_ii
dct_features = np.array([aurora_dct_ii(mel_features[i]) for i in range(M)])  # (M, N)

# Compute the correlations (after the DCT)
print("\n" + "=" * 70)
print("[After the DCT] correlations of the MFCC coefficients:")
print("=" * 70)
r_after = np.corrcoef(dct_features.T)  # (N, N) correlation matrix
print(f"r(0, 1) = {r_after[0, 1]:.4f}  (neighboring coefficients, should be close to 0 ✓)")
print(f"r(0, 2) = {r_after[0, 2]:.4f}  (2 apart, |r| clearly smaller — the DCT only approximately decorrelates)")
print(f"r(0, 5) = {r_after[0, 5]:.4f}  (5 apart, should be close to 0 ✓)")
print("\n[Structure of the covariance matrix]")
print("  off-diagonal entries mostly close to 0 → features approximately uncorrelated → the model works efficiently ✓")

# Visualize: compare the two covariance matrices
print("\n" + "=" * 70)
print("[Numbers compared] variance vs off-diagonal correlation")
print("=" * 70)
cov_before = np.cov(mel_features.T)
cov_after = np.cov(dct_features.T)

diag_before = np.diag(cov_before)
diag_after = np.diag(cov_after)
off_diag_before = np.abs(r_before[np.triu_indices_from(r_before, k=1)]).mean()
off_diag_after = np.abs(r_after[np.triu_indices_from(r_after, k=1)]).mean()

print(f"mel (before the DCT): mean variance {diag_before.mean():.4f}, mean off-diagonal |r| {off_diag_before:.4f}")
print(f"MFCC (after the DCT): mean variance {diag_after.mean():.4f}, mean off-diagonal |r| {off_diag_after:.4f}")
print(f"\nreduction in the off-diagonal correlation: {off_diag_before / off_diag_after:.1f}x (nice!)")

In [ ]:
# Demo: simulate correlated mel energies and look at the coefficients after the DCT
rng = np.random.default_rng(42)
# Use an AR(1) process to simulate highly correlated neighboring mel bins (phi=0.9)
N = 26
mel_energy = np.zeros(N)
mel_energy[0] = rng.standard_normal()
for i in range(1, N):
    mel_energy[i] = 0.9 * mel_energy[i-1] + 0.1 * rng.standard_normal()

# DCT by hand
k = np.arange(N); n = np.arange(N)
M = np.cos(np.pi * np.outer(k, 2*n + 1) / (2*N))
dct_out = M @ mel_energy

print("mel energy (first 5):", np.round(mel_energy[:5], 4))
print("DCT coefficients (all, absolute values):", np.round(np.abs(dct_out), 4))
print("energy concentration: share of total energy in the first 13 coefficients:",
      np.round(np.sum(dct_out[:13]**2) / np.sum(dct_out**2), 4))

## 4. ✏️ Implement `dct_ii(x)`

**Matrix form** (a single matrix–vector product):

```
D[k, n] = w(k) * cos(π·k·(2n+1) / (2N))
X = D @ x
```

**Four steps to implement it**:

| Step | Code | Notes |
|---|---|---|
| 1 | `N = len(x)` | signal length |
| 2 | build `D[k,n] = cos(π·k·(2n+1)/(2N))` | broadcast with `np.outer(k, 2n+1)` |
| 3 | multiply by the normalization weights `w(k)` | k=0: `1/√N`, k>0: `√(2/N)` |
| 4 | `return D @ x` | the matrix product outputs `(N,)` |

**Acceptance criteria**:
- `np.allclose(dct_ii(x), scipy.fft.dct(x, norm='ortho'), atol=1e-10)` passes
- orthogonality: `D @ D.T ≈ I` (checked in the checker cell)
- pure NumPy, no `import scipy`

> AURORA rule: implement the algorithm, no black boxes. `scipy.fft.dct` is used only for checking, never in the implementation.

## 4.1 Extra: two equivalent ways to use the scale vector

When you see `scale = [√(1/N), √(2/N), √(2/N), ...]`, you might wonder: how do I use this vector?

**Two mathematically equivalent implementations**:

### Way 1: compute M@x first, then multiply elementwise by scale
```python
X = (M @ x) * scale   # scale broadcasts against the (N,) vector
```
This is equivalent to:
```python
result = np.zeros(N)
for k in range(N):
    result[k] = scale[k] * (M @ x)[k]
```

### Way 2: multiply each row of M by scale first, then do the matrix product
```python
D = scale[:, None] * M   # row k of D = scale[k] * M[k, :]
X = D @ x
```

This is equivalent to:
```python
D = np.zeros((N, N))
for k in range(N):
    D[k, :] = scale[k] * M[k, :]
result = D @ x
```

**Mathematically**, the two are the same:
$$(\text{scale} \cdot (M \cdot x))[k] = \text{scale}[k] \cdot (M \cdot x)[k] = (scale[k] \cdot M[k, :]) \cdot x = D[k, :] \cdot x$$

**Implementation tips**:
- NumPy's matrix product is well optimized, so way 2 (`D = scale[:, None] * M; D @ x`) is often faster
- way 1 reads well and its logic is clear (project first, then rescale)

The code below checks that the two give exactly the same output:

In [ ]:
# Check that the two ways of using scale are equivalent
N = 8
x = np.random.default_rng(49).standard_normal(N)

# Build the cosine matrix and the weights
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
scale = np.full(N, np.sqrt(2.0 / N))
scale[0] = np.sqrt(1.0 / N)

# Way 1: do M@x first, then multiply by scale (elementwise)
X_method1 = (M @ x) * scale

# Way 2: multiply each row of M by scale first, then do the matrix product
D = scale[:, None] * M
X_method2 = D @ x

# Check
print("way 1 result:", np.round(X_method1, 6))
print("way 2 result:", np.round(X_method2, 6))
print("\ndifference between the two:", np.max(np.abs(X_method1 - X_method2)))
assert np.allclose(X_method1, X_method2, atol=1e-12), "the two ways should be exactly the same!"
print("✓ Check passed: the two ways are mathematically equivalent")

In [ ]:
def dct_ii(x: np.ndarray) -> np.ndarray:
    """DCT-II (orthonormal, norm='ortho'):
    X[k] = s_k * sum_{n=0}^{N-1} x[n] * cos(pi*(2n+1)*k/(2N))
    s_0 = sqrt(1/N),  s_k = sqrt(2/N)  for k > 0
    """
    # ✏️ TODO: compute N, k = np.arange(N), n = np.arange(N)
    # ✏️ TODO: build the cosine matrix M, shape (N, N)
    # ✏️ TODO: compute the scale vector (sqrt(1/N) for k=0, sqrt(2/N) for the rest)
    # ✏️ TODO: return (M @ x) * scale
    raise NotImplementedError("TODO: pure-NumPy orthonormal DCT-II: X[k] = s_k·Σ x[n]cos(π(2n+1)k/2N)")

In [ ]:
# Note: the reference implementation lives only in notebooks/5_audio_dsp/solutions/L49_dct_solutions.md

In [ ]:
# ✏️ Acceptance cell: checks *your own* dct_ii (no silent substitution)
# The standard matches what section 4 promised: pointwise atol=1e-10 against scipy.fft.dct(type=2, norm='ortho').
# AURORA rule: scipy serves here only as the "answer key" for comparison, never in the implementation.
from scipy.fft import dct as _scipy_dct

try:
    _rng_acc = np.random.default_rng(49)
    for _n in (4, 8, 26, 40):
        _x = _rng_acc.standard_normal(_n)
        _mine = np.asarray(dct_ii(_x))
        _ref = _scipy_dct(_x, type=2, norm="ortho")
        _err = float(np.max(np.abs(_mine - _ref)))
        assert _mine.shape == _ref.shape, f"N={_n}: output shape {_mine.shape} != {_ref.shape}"
        assert np.allclose(_mine, _ref, atol=1e-10), (
            f"N={_n}: your dct_ii vs scipy.fft.dct(type=2, norm='ortho') "
            f"max error {_err:.2e} > atol=1e-10 — check the normalization weights w(k) and the cos matrix"
        )
        print(f"  N={_n:2d} ✓ max error {_err:.2e}")
    print("✅ Accepted: your dct_ii matches scipy.fft.dct(type=2, norm='ortho') pointwise within atol=1e-10")
except (NotImplementedError, TypeError):
    print("⬜ dct_ii isn't implemented yet — finish the TODO in section 4 first, then come back and run this acceptance cell")

## 5. Parameter experiment: invertibility and truncated reconstruction

**Experiment A — checking invertibility**: the orthonormal DCT-II matrix $D$ satisfies $D D^T = I$ — not as an "assumption," but as a fact we've already checked in code: the orthogonality check in section 2 verified that the unweighted matrix $M$ is orthogonal ($M@M.T$ has $N$ or $N/2$ on the diagonal and 0 off it), and the section 2.3 demo verified $D@D.T \approx I$ directly for the weighted matrix $D$ (the orthogonality check cell near the end of this lesson asserts it again, to an error < 1e-10).

Because $D D^T = I$ has already been confirmed by running code, the inverse transform can simply use the transpose, with no matrix inverse needed:

```
x_rec = D.T @ X    # no matrix inverse, no scipy either
```

(Recall the definition of an "orthogonal matrix" from linear algebra: if a square matrix $D$ satisfies $D D^T = I$, then by definition the inverse of $D$ is $D^T$, i.e. $D^{-1} = D^T$. We've already verified $DD^T=I$ in the section 2 and section 2.3 checks, so here we're not "assuming" the matrix is invertible — we're citing a fact already verified.)

The `_idct_ii_ref` in the code below is a pure-NumPy reference inverse written straight from this formula.

**Experiment B — truncated reconstruction**: keep only the first `k_keep` DCT coefficients, set the rest to zero, invert, and watch how the reconstruction error changes with `k_keep`.
The test data is a smooth, correlated AR(1) vector (simulating mel energies whose neighbors are highly correlated, amplitude std≈0.21).
What to expect: with `k_keep=3` only the rough outline is left (RMSE≈0.09); at `k_keep=13` the RMSE≈0.03, already small relative to the signal's amplitude — and that's the basis for keeping the first 13 MFCC dimensions.
Note: energy concentration only holds for "smooth, correlated" vectors; try white noise instead (`rng.standard_normal(26)`) and cutting half the coefficients loses about half the energy — the DCT can't compress data that has no structure.

In [ ]:
# Experiments decoupled from the exercise: if your local dct_ii isn't finished yet, temporarily use aurora.audio.mfcc.dct_ii for the experiments below.
from aurora.audio.mfcc import dct_ii as aurora_dct_ii

# Choose an implementation: prefer the local dct_ii, otherwise use aurora's reference implementation
try:
    probe = dct_ii(np.array([1.0, 2.0]))
    _impl = dct_ii
    print("✓ Using your local dct_ii implementation for the experiments")
except (NameError, NotImplementedError, TypeError):
    _impl = aurora_dct_ii
    print("⚠️  Using the aurora.audio.mfcc.dct_ii reference implementation for the experiments")

# numpy-based inverse DCT-II reference
def _idct_ii_ref(X):
    # Inverse of ortho DCT-II: X = D @ x  ⟹  x = D.T @ X
    # where D[k,n] = scale[k] * cos(pi*k*(2n+1)/(2N)),
    # scale[0]=sqrt(1/N), scale[k>0]=sqrt(2/N)
    N = len(X)
    k = np.arange(N)
    n = np.arange(N)
    scale = np.full(N, np.sqrt(2.0 / N))
    scale[0] = np.sqrt(1.0 / N)
    D = scale[:, None] * np.cos(np.pi * np.outer(k, 2 * n + 1) / (2 * N))
    return D.T @ X

# Experiment on a 26-dimensional simulated mel energy: smooth AR(1) correlation (the same kind as the section 3 demo)
rng = np.random.default_rng(7)
N_mel = 26
x_mel = np.zeros(N_mel)
x_mel[0] = rng.standard_normal()
for i in range(1, N_mel):
    x_mel[i] = 0.9 * x_mel[i - 1] + 0.1 * rng.standard_normal()
print(f'simulated mel energy: std={x_mel.std():.3f} (smooth and correlated — that is what makes it compressible)')

X_dct = _impl(x_mel)

# Experiment A: exact inverse
x_rec_full = _idct_ii_ref(X_dct)
print('Experiment A — exact reconstruction error (should be ≈0):', np.max(np.abs(x_rec_full - x_mel)))

# Experiment B: truncated reconstruction
print('\nExperiment B — truncated reconstruction error:')
for k_keep in [3, 6, 13]:
    X_trunc = X_dct.copy()
    X_trunc[k_keep:] = 0.0
    x_trunc_rec = _idct_ii_ref(X_trunc)
    rmse = np.sqrt(np.mean((x_trunc_rec - x_mel) ** 2))
    print(f'  k_keep={k_keep:2d} → RMSE={rmse:.4f}')

## Wrap-up

`dct_ii(x)` builds an N×N cosine matrix and does a matrix product, outputting a real-valued vector of cepstral coefficients the same length as the input.
The function is the third step of the MFCC pipeline in the `aurora.audio.mfcc` module: it takes the log-mel energy spectrum, and its low-order output coefficients form the MFCCs we know.
The truncation experiment shows that for a smooth, correlated mel-like vector, the first 13 coefficients already reconstruct the whole curve approximately (the RMSE is only about a tenth of the signal's amplitude). That's why MFCC-13 became the standard feature size for speech recognition.
Next lesson (L50) embeds `dct_ii` into the full MFCC pipeline: STFT → mel filtering → log → DCT → truncation, processing real audio frames end to end.

## ✏️ Closed-book derivation check cell — DCT-II orthonormalization

**Rule: close every cell above and do the derivation below from memory alone.**

**Problem**: the transform matrix $D$ of the orthonormal DCT-II satisfies $D D^T = I$.

Its weights are:
$$w_k = \begin{cases} \sqrt{1/N} & k = 0 \\ \sqrt{2/N} & k > 0 \end{cases}$$

1. Write the full expression for the matrix element $D_{k,n}$ (including the weight $w_k$)
2. Verify that row $k=0$ and row $k=1$ are orthogonal (inner product 0): compute $\sum_{n=0}^{N-1} D_{0,n} D_{1,n}$ by hand

(Write your derivation here...)

## Extra: background for the closed-book problem: a quick trig-identity reference

When you derive "row k=0 and row k=1 are orthogonal," you'll run into a sum like this:

$$\sum_{n=0}^{N-1} \cos\left(\frac{\pi k(2n+1)}{2N}\right) = ?$$

This calls for the **complex unit-circle method** (we already used it once when deriving $\sum\cos^2$ in section 1.5); now we extend the same method to the more general case — no longer requiring $k_1=k_2$.

**Don't worry: the method is exactly the same as in section 1.5, with just one extra product-to-sum step.**

**Step one: use the product-to-sum identity to split a product of two cosines into a sum of two cosines**

The school formula:
$$\cos A\cos B = \frac{1}{2}\big[\cos(A-B) + \cos(A+B)\big]$$

Let $A = \dfrac{\pi k_1(2n+1)}{2N}$ and $B = \dfrac{\pi k_2(2n+1)}{2N}$, and substitute:

$$\cos A \cos B = \frac{1}{2}\cos\!\left(\frac{\pi(k_1-k_2)(2n+1)}{2N}\right) + \frac{1}{2}\cos\!\left(\frac{\pi(k_1+k_2)(2n+1)}{2N}\right)$$

Summing over n, the two terms are cosine sums over angles in arithmetic progression, with common differences $\frac{\pi(k_1-k_2)}{N}$ and $\frac{\pi(k_1+k_2)}{N}$.

**Step two: apply the lemma from section 1.5**

The same trick as in section 1.5 (use Euler's formula to turn the cosine sum into a geometric series with ratio $e^{i\pi m/N}$, then take the real part) shows: for any integer $m$ that isn't a multiple of $2N$, $\sum_{n=0}^{N-1}\cos\!\left(\dfrac{\pi m(2n+1)}{2N}\right) = 0$. (Section 1.5 did the even case $m=2k$, where the ratio $e^{i2\pi k/N}$ goes round a whole number of turns and everything cancels tip to tail.)

Apply this lemma to the two terms, "$m=k_1-k_2$" and "$m=k_1+k_2$":

- If $k_1\neq k_2$ (both in $0,\dots,N-1$): $|k_1-k_2|$ and $k_1+k_2$ lie between 1 and $2N-3$, so neither is a multiple of $2N$; both terms are 0, and **the whole sum is 0**.
- If $k_1=k_2=k\neq 0$: the first term becomes $\cos(0)=1$ summed N times $=N$, and the second term ($m=2k$, between 2 and $2N-2$) is still 0. Multiplying by the $\frac12$ in front gives $N/2$ — exactly the result from section 1.5.
- If $k_1=k_2=0$: both terms are $\cos(0)=1$, and the sum is $\frac{N+N}{2} = N$.

That's the full derivation: not a conclusion handed down from nowhere, but two steps — "product-to-sum + the geometric-series lemma from section 1.5." **In the closed-book exercise** you don't need to re-derive all of this yourself (it goes a bit beyond school maths),
but you do need to **understand why it leads to orthogonality**.

### Key identities (summary of the results)

1. **Inner products of cosines (the basis of orthogonality)**:
   $$\sum_{n=0}^{N-1} \cos\left(\frac{\pi k_1(2n+1)}{2N}\right) \cos\left(\frac{\pi k_2(2n+1)}{2N}\right) = \begin{cases}
   N/2 & \text{if } k_1 = k_2 \neq 0 \\
   N & \text{if } k_1 = k_2 = 0 \\
   0 & \text{if } k_1 \neq k_2
   \end{cases}$$

2. **Sum of squared cosines**:
   $$\sum_{n=0}^{N-1} \cos^2\left(\frac{\pi k(2n+1)}{2N}\right) = \frac{N}{2} \quad (k > 0)$$

These two identities guarantee that the DCT matrix is orthogonal.

### A quick numerical check

In [ ]:
# Numerical check of the trig identities
N = 8

print("=" * 70)
print("Numerical check of the trig identities:")
print("=" * 70)

# Identity 1: inner products of cosines
print("\n[Identity 1] inner products of cos (should satisfy orthogonality):")
for k1 in range(4):
    for k2 in range(4):
        n = np.arange(N)
        cos1 = np.cos(np.pi * k1 * (2*n + 1) / (2*N))
        cos2 = np.cos(np.pi * k2 * (2*n + 1) / (2*N))
        inner_product = np.sum(cos1 * cos2)

        if k1 == k2 == 0:
            expected = N
        elif k1 == k2 != 0:
            expected = N / 2
        else:
            expected = 0

        match = "✓" if np.isclose(inner_product, expected, atol=1e-10) else "✗"
        print(f"  k1={k1}, k2={k2}: computed {inner_product:7.4f}, expected {expected:5.1f}  {match}")

# Identity 2: sum of squared cosines
print("\n[Identity 2] sum of cos² (should be N/2):")
for k in range(1, 4):
    n = np.arange(N)
    cos_k = np.cos(np.pi * k * (2*n + 1) / (2*N))
    sum_cos_sq = np.sum(cos_k**2)
    expected = N / 2
    match = "✓" if np.isclose(sum_cos_sq, expected, atol=1e-10) else "✗"
    print(f"  k={k}: computed {sum_cos_sq:7.4f}, expected {expected:5.1f}  {match}")

In [ ]:
# Check: does the DCT matrix satisfy D @ D.T ≈ I (orthogonality)?
import sys
sys.path.insert(0, 'src')

# If dct_ii isn't implemented yet, use aurora's reference implementation; otherwise use the local one
try:
    probe = dct_ii(np.array([1.0, 2.0]))
    print("✓ Local dct_ii is implemented; using it for the orthogonality check")
    impl = dct_ii
except (NameError, NotImplementedError, TypeError):
    print("⚠️  Local dct_ii isn't implemented; using the aurora.audio.mfcc.dct_ii reference implementation")
    from aurora.audio.mfcc import dct_ii as impl

N = 16
# Feed in each row of the identity matrix → build the DCT transform matrix
D_rows = []
for n in range(N):
    row = np.zeros(N)
    row[n] = 1.0  # the n-th standard basis vector
    dct_row = impl(row)
    D_rows.append(dct_row)
D = np.array(D_rows)  # (N, N)

# Check orthogonality
gram = D @ D.T
err = np.abs(gram - np.eye(N)).max()
assert err < 1e-10, f"D @ D.T max error {err:.2e} — not orthogonal"
print(f"✅ DCT-II orthogonality check passed (error = {err:.2e})")

In [ ]:
# ✏️ Self-review for this lesson
l49_review = {
    "dct_formula":            None,  # remember D[k,n]=w(k)·cos(πk(2n+1)/2N), w(0)=1/√N? True/False
    "dct_ii_implemented":     None,  # dct_ii implemented and passes the scipy ortho comparison (atol<1e-10)? True/False
    "decorrelation_purpose":  None,  # understand that the DCT decorrelates, rotating correlated mel energies into cepstral coordinates? True/False
    "mfcc13_reason":          None,  # understand why the first 13 dimensions are kept (low-order coefficients = phoneme information)? True/False
    "whiteboard_passed":      None,  # closed-book whiteboard derivation of DCT orthonormalization passed? True/False
}

unfilled = [k for k, v in l49_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l49_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L49 fully cleared! On to L50: the full MFCC pipeline')

---

→ **Next lesson** · [L50 · The full MFCC pipeline](L50_mfcc.ipynb)

> Next lesson covers **the full MFCC pipeline**: signal → STFT → mel → log → DCT, checking the output shape at every step.